# 04 — Redacted Frame Extraction (serverless)

Extracts one frame from a shoplifting video, detects faces, and pixelates them before saving a PNG to a UC Volume. Use it for slides, blog posts, or customer demos: the Kaggle footage is public, but the people in it never agreed to appear next to a "shoplifting" label.

**Compute:** serverless CPU. No GPU, no `apt-get`.

Differences from the classic version (`3_Video_Batch_Inference_Ray_Core`):
* System `ffmpeg` (installed with `apt-get`) is replaced by [PyAV](https://pyav.basswood-io.com/), which bundles FFmpeg as a pip wheel.
* MediaPipe is replaced by OpenCV's built-in YuNet face detector. MediaPipe pins `protobuf<5` and `numpy<2`, which breaks the serverless Python kernel.
* Catalog, schema, and volume defaults match notebooks `00`–`03` in this folder.

The second detection pass (Haar cascades), pixelation, and the no-face fallback blur are unchanged.

In [0]:
%pip install -q "av==14.0.1" "opencv-python-headless==4.10.0.84"
# MediaPipe is not used: it pins protobuf<5 and numpy<2, which breaks the serverless Python kernel.
dbutils.library.restartPython()


In [0]:
dbutils.widgets.text("CATALOG", "main", label="CATALOG")
dbutils.widgets.text("SCHEMA", "vlm_video", label="SCHEMA")  # same as notebook 00
dbutils.widgets.text("VOLUME", "transcribe-video", label="VOLUME")
dbutils.widgets.text("VIDEO_FILE", "Shoplifting028_x264.mp4", label="VIDEO_FILE")
dbutils.widgets.text("VIDEO_CATEGORY", "shoplifting", label="VIDEO_CATEGORY")
dbutils.widgets.text("FRAME_AT_SECONDS", "20", label="FRAME_AT_SECONDS")
dbutils.widgets.text("OUTPUT_FILENAME", "Shoplifting028_redacted_frame.png", label="OUTPUT_FILENAME")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
VOLUME = dbutils.widgets.get("VOLUME")
VIDEO_FILE = dbutils.widgets.get("VIDEO_FILE")
VIDEO_CATEGORY = dbutils.widgets.get("VIDEO_CATEGORY")
FRAME_AT_SECONDS = float(dbutils.widgets.get("FRAME_AT_SECONDS"))
OUTPUT_FILENAME = dbutils.widgets.get("OUTPUT_FILENAME")

video_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/raw_video/CCTV/data/{VIDEO_CATEGORY}/{VIDEO_FILE}"
output_dir = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/redacted_frames"
output_path = f"{output_dir}/{OUTPUT_FILENAME}"

print(f"Source video:  {video_path}")
print(f"Output:        {output_path}")
print(f"Frame at:      {FRAME_AT_SECONDS}s")

import importlib.metadata as md
print({p: md.version(p) for p in ["numpy", "protobuf", "av", "opencv-python-headless"]})

In [0]:
import av

with av.open(video_path) as container:
    stream = container.streams.video[0]
    # Seek to the keyframe before the target, then decode forward to the exact time.
    container.seek(int(FRAME_AT_SECONDS / stream.time_base), stream=stream, backward=True)
    img = None
    for frame in container.decode(stream):
        img = frame.to_ndarray(format="bgr24")  # OpenCV channel order
        frame_time = frame.time
        if frame_time is not None and frame_time >= FRAME_AT_SECONDS:
            break

if img is None:
    raise RuntimeError(f"No frame decoded from {video_path} at {FRAME_AT_SECONDS}s")
h, w = img.shape[:2]
print(f"Extracted frame at {frame_time:.2f}s ({w}x{h})")

In [0]:
import os
import cv2
import urllib.request

# CCTV frames are tiny (often 320x240), so faces are only a few pixels wide.
# Detect on an upscaled copy, then map the boxes back to the original frame.
UPSCALE = max(1, round(960 / max(w, h)))
det_img = cv2.resize(img, (w * UPSCALE, h * UPSCALE), interpolation=cv2.INTER_CUBIC) if UPSCALE > 1 else img
dh, dw = det_img.shape[:2]
print(f"Detecting on {dw}x{dh} (upscale x{UPSCALE})")

# --- Pass 1: OpenCV YuNet (CNN face detector built into OpenCV; replaces MediaPipe) ---
YUNET_URL = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
yunet_path = "/tmp/face_detection_yunet_2023mar.onnx"
if not os.path.exists(yunet_path):
    urllib.request.urlretrieve(YUNET_URL, yunet_path)
yunet = cv2.FaceDetectorYN.create(yunet_path, "", (dw, dh), score_threshold=0.3)
_, dets = yunet.detect(det_img)
faces_dnn = [tuple(int(v / UPSCALE) for v in d[:4]) for d in dets] if dets is not None else []
print(f"YuNet found {len(faces_dnn)} face(s)")

# --- Pass 2: OpenCV Haar (frontal + profile) ---
gray = cv2.cvtColor(det_img, cv2.COLOR_BGR2GRAY)
cascade_paths = [
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml",
    cv2.data.haarcascades + "haarcascade_profileface.xml",
]
faces_cv = []
for path in cascade_paths:
    cascade = cv2.CascadeClassifier(path)
    detections = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(20, 20))
    faces_cv.extend(tuple(int(v / UPSCALE) for v in b) for b in detections)
print(f"OpenCV Haar found {len(faces_cv)} face(s)")

all_boxes = list(faces_dnn) + [tuple(b) for b in faces_cv]
print(f"Total face regions to pixelate: {len(all_boxes)}")

def pixelate_region(image, x, y, bw, bh, mosaic_size=12):
    x = max(0, x); y = max(0, y)
    bw = min(image.shape[1] - x, bw); bh = min(image.shape[0] - y, bh)
    if bw <= 0 or bh <= 0:
        return
    region = image[y:y+bh, x:x+bw]
    small = cv2.resize(region, (max(1, bw // mosaic_size), max(1, bh // mosaic_size)), interpolation=cv2.INTER_LINEAR)
    pixelated = cv2.resize(small, (bw, bh), interpolation=cv2.INTER_NEAREST)
    image[y:y+bh, x:x+bw] = pixelated

margin = 0.3
for (x, y, bw, bh) in all_boxes:
    cx, cy = x + bw / 2, y + bh / 2
    new_bw = int(bw * (1 + margin))
    new_bh = int(bh * (1 + margin))
    new_x = int(cx - new_bw / 2)
    new_y = int(cy - new_bh / 2)
    pixelate_region(img, new_x, new_y, new_bw, new_bh, mosaic_size=14)

In [0]:
if not all_boxes:
    print("WARNING: no face detected — applying conservative blur to upper third of frame")
    upper_h = h // 3
    upper = img[0:upper_h, :]
    blurred = cv2.GaussianBlur(upper, (51, 51), 0)
    img[0:upper_h, :] = blurred

In [0]:
ann_text = "Face redacted (auto-detected, pixelated)"
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 0.55
thickness = 1
margin_px = 8
# Shrink the label until it fits small CCTV frames (320 px wide).
while font_scale > 0.25 and cv2.getTextSize(ann_text, font, font_scale, thickness)[0][0] > w - 2 * margin_px:
    font_scale -= 0.05
(text_w, text_h), baseline = cv2.getTextSize(ann_text, font, font_scale, thickness)
x = margin_px
y = h - margin_px
overlay = img.copy()
cv2.rectangle(overlay, (x - 4, y - text_h - 4), (x + text_w + 4, y + baseline), (0, 0, 0), -1)
img = cv2.addWeighted(overlay, 0.55, img, 0.45, 0)
img = cv2.putText(img, ann_text, (x, y), font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

In [0]:
import os
os.makedirs(output_dir, exist_ok=True)
ok = cv2.imwrite(output_path, img, [cv2.IMWRITE_PNG_COMPRESSION, 6])
if not ok:
    raise RuntimeError(f"Failed to write {output_path}")
size = os.path.getsize(output_path)
print(f"Wrote redacted frame to {output_path} ({size:,} bytes, {w}x{h})")
display(spark.read.format("binaryFile").load(output_path).select("path", "length"))

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(cv2.imread(output_path), cv2.COLOR_BGR2RGB))
plt.title(OUTPUT_FILENAME)
plt.axis("off")
plt.show()
print("Detection is automatic: check by eye that every face is covered before using this frame publicly.")